# Module Summary — Project 1: Data Workflow
### STATS19 Road Safety Collision Data

*Editable draft — content is grounded in the project decision log and the executed `data_workflow.ipynb`. Sections and headings follow the Task 10 requirements exactly.*

## Overview

This report accompanies `data_workflow.ipynb`, a Pandas- and Matplotlib-based workflow analysing UK road traffic collision severity using **STATS19 Road Safety Collision Data**, the Department for Transport's official collision database ([gov.uk Road Safety Data](https://www.gov.uk/government/statistics/road-safety-data)). The notebook ingests, cleans and explores the DfT's pre-curated "last five years" collision extract to examine how collision severity relates to time (year, hour, day of week, month) and posted speed limit. No model training was carried out at this stage; the project instead establishes a reproducible foundation for later machine learning and deep learning work.

## Dataset Description

The collision-level STATS19 table records 513,801 collisions across the years 2021–2025, with 44 columns covering location, severity, road environment, and contributory conditions. Only the collision table was used, rather than the related vehicle- and casualty-level tables, keeping the analysis focused on a single, well-understood table appropriate for a pre-modelling workflow. The full 44-column table was inspected during ingestion, with dtypes and summary statistics checked against the whole schema, before the working dataframe was narrowed to the nine columns this analysis actually needed: collision identifiers, year, date, time, day of week, collision severity, speed limit, and vehicle/casualty counts. The key variable throughout is `collision_severity`, the legacy three-category classification (Fatal, Serious, Slight), chosen over the newer six-category `enhanced_collision_severity` because the five-year window spans the 2023 point at which the enhanced scheme was introduced, and a single, consistent definition was judged more important than the additional granularity.

## Workflow Description

- **Ingestion:** the raw CSV was loaded in full (44 columns) with explicit category dtypes applied to detected coded fields, so that summary statistics and dtype inspection reflected the whole schema before any column was dropped. The working dataframe was then narrowed to the nine columns this analysis needed.
- **Cleaning:** two functions were applied — one converting STATS19's `-1` missing/unknown sentinel to a proper null, and one decoding coded categorical fields (severity, day of week) by merging against lookup tables built from the official DfT data guide.
- **Exploratory analysis:** a single, parameterised function grouped the cleaned data by year, speed limit, hour, day of week and month in turn, returning both raw counts and severity proportions for each grouping.
- **Visualisation:** five figures were produced — two proportional stacked bar charts (severity by year, severity by speed limit) and three polar "rose" charts (severity by hour, day of week and month) — each with a title and labelled axes.
- **Summary:** a closing markdown section in the notebook sets out what was learned, the key patterns, the limitations and assumptions, and the most surprising findings.

## Key Decisions and Assumptions

**Cleaning choices.** Two cleaning functions were required and used. The first converts STATS19's `-1` missing/unknown sentinel to a proper null value; left as a literal `-1`, this sentinel is silently counted as valid data by pandas' native missing-value tools, understating true missingness. The second decodes coded categorical fields (collision severity and day of week) by merging the cleaned data against lookup tables built from the official DfT data guide, rather than inferring what each code means, so that the resulting labels are grounded in the authoritative source. A related helper function derives hour-of-day and month from the raw time and date columns; it is kept separate from both cleaning functions and the EDA function so that each function has a single, clearly testable responsibility, following general code-quality guidance for research code (The Turing Way Community, 2022).

**EDA focus.** Rather than one monolithic function, a single small, parameterised function (`severity_breakdown`) was written and called once per grouping variable — year, speed limit, hour, day of week, and month — each time returning both raw counts and severity proportions. This grouped, comparative approach follows the exploratory data analysis tradition of examining data from multiple angles, using both graphical and quantitative summaries, to see what patterns are present before any formal modelling is attempted (Heckert & Filliben, 2003). Counts were kept alongside proportions specifically because proportions alone can make a small, sparse group look artificially alarming or reassuring; for example, a group of two collisions that happen to both be fatal would show as a 100% fatal rate despite carrying no statistical weight.

**What each plot was designed to show.** Figures 1 and 2 use proportional stacked bars to show how the severity mix shifts across an ordered variable (year, then speed limit). Figures 3–5 use polar "rose" charts, a well-established way of displaying cyclical data — such as hourly, daily, or monthly traffic-collision patterns — so that the periodic structure of the variable is visible directly in the chart's layout, rather than being artificially flattened onto a linear axis with an arbitrary start and end point (Chen, 2022). Plotting collision volume as wedge length and severity mix as stacked colour within each wedge means both patterns are visible in a single chart, rather than only volume or only proportions.

**Reproducibility/workflow decision.** A dedicated virtual environment was created before any package was installed, and only the packages actually needed (numpy, pandas, matplotlib, seaborn, openpyxl, jupyter) were installed into it; `requirements.txt` was then generated from inside that clean environment with `pip freeze`. Danchev (2022) notes that differences between a project's intended dependencies and the packages already present in a given computing environment can introduce errors when work is reproduced elsewhere; generating `requirements.txt` from a clean, isolated environment, rather than from a general-purpose base install, keeps the dependency list accurate to what the project actually needs.

## Results and Interpretation

Across the five-year extract (2021–2025, 513,801 collisions), Slight collisions make up roughly three-quarters of the total and Fatal collisions are rare, at around 1.5% — a pattern visible in **Figure 1**. The Serious share drifted gently upward over the period, from around 21% in 2021 to around 25% in 2025, while the Fatal share stayed essentially flat.

**Figure 2** shows the clearest pattern in the dataset: the combined Fatal-and-Serious share rises steadily with speed limit, from around 20% at 20mph to around 34% at 60mph, then drops back to around 25% at 70mph. Since 70mph limits in Great Britain apply almost exclusively to motorways and grade-separated dual carriageways, which lack oncoming traffic and junctions, this suggests speed limit is acting partly as a proxy for road environment rather than purely for speed.

**Figures 3–5** show a consistent "volume versus risk" pattern across all three cyclical breakdowns. Collision volume peaks during commuting hours and on Fridays (Figures 3 and 4), and is fairly flat across the months with a dip in February (Figure 5, largely a day-count effect given February's shorter length). However, the collisions that do occur outside these peak periods — overnight, at weekends (particularly Sunday), and in the summer months — consistently carry a higher Fatal-and-Serious share despite their lower volume, a pattern that runs against a naive assumption that worse weather or lower visibility alone drives worse outcomes.

## Responsible Practice (Bias and Data Quality)

The main cleaning-stage bias risk identified was around how missing values are handled. STATS19 encodes unknown or missing values on coded fields as a literal `-1` rather than a null; if these rows had been dropped outright rather than converted to a proper null, an entire stratum of the data could have been silently discarded, biasing the remaining sample toward whatever collisions happened to have complete records. A related risk is imputing a "typical" value in place of a genuine unknown, which would mask real missingness as if it were ordinary data and produce misleadingly confident conclusions. Rubin (1976) shows that whether it is appropriate to ignore the process that generated missing data depends on the pattern of that missingness, meaning naive cleaning approaches — dropping or imputing without considering why data are missing — can shift results and introduce bias rather than removing it. This is why the cleaning function used here nulls the sentinel explicitly, preserving the missingness as information, rather than dropping or imputing it.

A further limitation is one of scope, not correctness: the `-1` sentinel check (and the whole cleaning/EDA/visualisation pipeline) was run only on the nine analysis columns, not across all 44 raw columns, so the patterns above say nothing about missingness or bias in fields such as weather conditions, road surface, or junction detail, which were inspected but not analysed here and could plausibly confound the speed-limit and time-of-day patterns reported above.

## Reproducibility

The project is reproducible from a fresh clone: cloning the repository, creating a virtual environment, installing from `requirements.txt` (itself generated with `pip freeze` from that same clean environment, as discussed above and supported by Danchev, 2022), and running `data_workflow.ipynb` top to bottom reproduces the full workflow with no additional download step, since the raw data files are expected at fixed local paths rather than fetched over the network. Version control followed a simple branching workflow: a `dev` branch was used for the incremental, per-task commits that build up the notebook, cleaning functions, visualisations and README, with `main` kept in step as each task was completed — giving the repository multiple commits and more than one branch.

## Sources and Citations (Required)

*All sources below are freely accessible online (no paywall or institutional login required); links are given for each.*

Chen, K.-T. (2022). *It's a wrap! Visualisations that wrap around cylindrical, toroidal, or spherical topologies* [Doctoral thesis, Monash University]. Monash University Bridges Repository. https://doi.org/10.26180/20723092.v1

Danchev, V. (2022). Reproducible data science with Python: An open learning resource. *Journal of Open Source Education, 5*(56), 156. https://doi.org/10.21105/jose.00156

Heckert, N. A., & Filliben, J. J. (2003). *NIST/SEMATECH e-handbook of statistical methods, Chapter 1: Exploratory data analysis.* National Institute of Standards and Technology. https://www.itl.nist.gov/div898/handbook/eda/eda.htm

Rubin, D. B. (1976). Inference and missing data. *Biometrika, 63*(3), 581–592. https://dash.harvard.edu/handle/1/3408223

The Turing Way Community. (2022). Code quality. In *The Turing Way: A handbook for reproducible, ethical and collaborative research.* https://book.the-turing-way.org/reproducible-research/code-quality.html